# Sentinel-X — Camera feed

Step 1 only: open the webcam and show the live feed.

- Press **q** in the video window to stop (or use the notebook's stop button).
- Nothing is saved.
- If the window does not open, change `SOURCE` to `1`.

In [1]:
%pip install opencv-python
%pip install mediapipe

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\grym\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\grym\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import urllib.request

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

## Settings

## Live feed

In [3]:
import time
import cv2

SOURCE = 0        # 0 = default webcam, 1 = second camera, or a video file path
WIDTH  = 640      # display width in px
WINDOW = "Sentinel-X camera (q or X to quit)"


def window_closed():
    """True once the user clicked the X in the window's title bar."""
    try:
        visible = cv2.getWindowProperty(WINDOW, cv2.WND_PROP_VISIBLE)
        autosize = cv2.getWindowProperty(WINDOW, cv2.WND_PROP_AUTOSIZE)
        return visible < 1 or autosize < 0       # -1 = window no longer exists
    except cv2.error:
        return True                              # window destroyed -> error -> closed


def show_feed(process=None, on_key=None):
    """
    Affiche le flux caméra.
      process(frame) -> frame : traitement appliqué à chaque image (optionnel)
      on_key(key)            : appelé à chaque touche pressée (optionnel)
    """
    cap = cv2.VideoCapture(SOURCE)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)      # ask the camera for full HD
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)          # always the freshest frame
    if not cap.isOpened():
        print("Cannot open the camera. Try another SOURCE (1, 2...) or close apps using it.")
        return

    n, fps, t_prev = 0, 0.0, time.perf_counter()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("No frame received (camera lost or video ended).")
                break
            n += 1
            h, w = frame.shape[:2]
            frame = cv2.resize(frame, (WIDTH, int(h * WIDTH / w)))

            if process:                              # <- les étapes suivantes se branchent ici
                frame = process(frame)

            t_now = time.perf_counter()
            fps = 0.9 * fps + 0.1 / max(t_now - t_prev, 1e-6) if n > 1 else 0.0
            t_prev = t_now
            cv2.putText(frame, f"{fps:.0f} fps", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

            cv2.imshow(WINDOW, frame)
            key = cv2.waitKey(1) & 0xFF              # also processes the X click
            if key == ord("q"):
                break
            if on_key and key != 255:
                on_key(key)
            if n > 2 and window_closed():            # checked BEFORE the next imshow,
                break                                # otherwise the window reopens
    except KeyboardInterrupt:
        print("Stopped.")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        cv2.waitKey(1)                           # lets macOS really close the window
        print(f"Camera released. {n} frames shown.")


show_feed()     # flux caméra simple, sans traitement

Camera released. 658 frames shown.


In [4]:
# %pip install -U mediapipe     # à lancer une fois si besoin

# ---------- RÉGLAGES ----------
MAX_FACES  = 10      # nombre de visages suivis en même temps
CONF       = 0.3     # seuil de confiance (0.5 avant -> plus sensible)
WIDTH      = 1280    # résolution de traitement (écrase le 640 de la cellule du flux)
TILE_FRAC  = 0.6     # taille d'une tuile = 60 % de la hauteur de l'image (plus petit = voit plus loin, plus lent)
TILE_EVERY = 3       # passe "tuiles" toutes les N images (résultat gardé entre deux passes)
# ------------------------------

MODEL_PATH = "face_landmarker.task"
MODEL_URL  = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
              "face_landmarker/float16/1/face_landmarker.task")

if not os.path.exists(MODEL_PATH):
    print("Téléchargement du modèle...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

Conn = vision.FaceLandmarksConnections


def draw_connections(frame, pts, connections, color, thickness=1):
    for c in connections:
        cv2.line(frame, pts[c.start], pts[c.end], color, thickness, cv2.LINE_AA)


def bbox(pts):
    xs, ys = zip(*pts)
    return min(xs), min(ys), max(xs), max(ys)


def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0


class FaceLandmarker:
    """Détecte les visages (image entière + tuiles zoomées pour les visages lointains)."""

    def __init__(self, max_faces=MAX_FACES):
        def make(mode):
            return vision.FaceLandmarker.create_from_options(
                vision.FaceLandmarkerOptions(
                    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
                    running_mode=mode,
                    num_faces=max_faces,
                    min_face_detection_confidence=CONF,
                    min_face_presence_confidence=CONF,
                    min_tracking_confidence=CONF,
                ))
        self.landmarker = make(vision.RunningMode.VIDEO)       # image entière, avec suivi
        self.tile_landmarker = make(vision.RunningMode.IMAGE)  # tuiles, sans suivi
        self.show_mesh = False                                 # touche m
        self.show_dots = False                                 # touche l
        self.n = 0
        self.tile_cache = []                                   # visages trouvés par les tuiles

    def _tiles(self, w, h):
        side = int(h * TILE_FRAC)
        step = int(side * 0.75)                                # 25 % de recouvrement
        xs = list(range(0, max(w - side, 0) + 1, step))
        ys = list(range(0, max(h - side, 0) + 1, step))
        if xs[-1] != max(w - side, 0): xs.append(max(w - side, 0))
        if ys[-1] != max(h - side, 0): ys.append(max(h - side, 0))
        return [(x, y, min(side, w), min(side, h)) for y in ys for x in xs]

    def _scan_tiles(self, rgb, w, h):
        found = []
        for x0, y0, tw, th in self._tiles(w, h):
            crop = np.ascontiguousarray(rgb[y0:y0+th, x0:x0+tw])
            res = self.tile_landmarker.detect(
                mp.Image(image_format=mp.ImageFormat.SRGB, data=crop))
            for face in res.face_landmarks:
                pts = [(int(p.x * tw) + x0, int(p.y * th) + y0) for p in face]
                bb = bbox(pts)
                if all(iou(bb, bbox(f)) < 0.3 for f in found):   # évite les doublons entre tuiles
                    found.append(pts)
        return found

    def __call__(self, frame):
        h, w = frame.shape[:2]
        self.n += 1
        rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # 1) image entière (suivi)
        ts_ms = int(time.perf_counter() * 1000)
        result = self.landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts_ms)
        all_faces = [[(int(p.x * w), int(p.y * h)) for p in face]
                     for face in result.face_landmarks]

        # 2) tuiles zoomées (visages lointains), toutes les TILE_EVERY images
        if self.n % TILE_EVERY == 1:
            self.tile_cache = self._scan_tiles(rgb, w, h)
        for pts in self.tile_cache:
            bb = bbox(pts)
            if all(iou(bb, bbox(f)) < 0.3 for f in all_faces):
                all_faces.append(pts)

        for pts in all_faces:
            if self.show_mesh:
                draw_connections(frame, pts, Conn.FACE_LANDMARKS_TESSELATION, (90, 90, 90))
            draw_connections(frame, pts, Conn.FACE_LANDMARKS_CONTOURS, (255, 255, 255))
            draw_connections(frame, pts, Conn.FACE_LANDMARKS_LEFT_IRIS, (0, 255, 255))
            draw_connections(frame, pts, Conn.FACE_LANDMARKS_RIGHT_IRIS, (0, 255, 255))
            if self.show_dots:
                for x, y in pts:
                    cv2.circle(frame, (x, y), 1, (0, 255, 255), -1)
            x1, y1, x2, y2 = bbox(pts)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(frame, f"faces: {len(all_faces)}", (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        return frame

    def on_key(self, key):
        if key == ord("m"):
            self.show_mesh = not self.show_mesh
        elif key == ord("l"):
            self.show_dots = not self.show_dots

    def close(self):
        self.landmarker.close()
        self.tile_landmarker.close()


faces = FaceLandmarker()
try:
    show_feed(process=faces, on_key=faces.on_key)    # q ou X = quitter, m = maillage, l = points
finally:
    faces.close()

Camera released. 577 frames shown.
